# <span style="color:darkblue"> Project 1 - Scraping the Healthy Food Policy Project Database </span>

<font size = "3">

- Website: https://healthyfoodpolicyproject.org/policy-database (permission obtained for this health-policy website)
- Data collected: jurisdiction name, jurisdiction type, state, policy category, summary text, and policy URL
- The site's ``` robots.txt ``` allows policy pages and asks for a 10-second delay between requests, so the code waits 10 seconds after each page

# <span style="color:darkblue"> I. Import Packages </span>

In [ ]:
# Manage datasets
import pandas as pd

# Work with time data
import time

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Automate navigating within the browser (Selenium)
from selenium import webdriver

# <span style="color:darkblue"> II. Running Chrome from Python </span>

<font size = "5">

Initialize web driver and open the website

<font size = "3">

- This opens a Google Chrome window (do not close it)

In [ ]:
options = webdriver.ChromeOptions()

driver = webdriver.Chrome(options=options)

starting_url = 'https://healthyfoodpolicyproject.org/policy-database'

driver.get(starting_url)

# <span style="color:darkblue"> III. Find the Policy Detail Pages </span>

<font size = "5">

Find the "View Details" links by tag + attribute value

<font size = "3">

- Each policy card has a button container ``` <div class = "policy-card-button-wrap"> ``` <br>
and the link to the policy is the tag ``` a ``` inside it
- Use ```.find_elements()``` (with a plural "s") to find multiple elements
- Extract the web address with ``` .get_attribute("href") ``` and store the addresses <br>
in a list <b>before</b> leaving the listing page

In [ ]:
list_links = driver.find_elements('xpath','//div[@class = "policy-card-button-wrap"]/a')

# Count how many elements are extracted
num_links = len(list_links)
print(num_links)

# Store the web address of each policy
list_urls = []
for i in range(num_links):
    list_urls.append(list_links[i].get_attribute("href"))

# Display the first policy web addresses
list_urls[0:3]

# <span style="color:darkblue"> IV. Test on a Single Policy </span>

<font size = "5">

Make sure the code works on a single element before using a loop

<font size = "3">

- Click the "View Details" button of the first policy
- Each field sits in a box ``` <div class = "policy-overview-grid-item"> ``` with a label <br>
(for example ``` <strong>State</strong> ```) followed by the value
- ```.text``` returns the label and the value, so we remove the label
- A field can be empty on the website. In that case we store an empty string

In [ ]:
# Extract the value of one labelled field on the policy page
def get_field(label):
    xpath = '//div[contains(@class,"policy-overview-grid-item")][strong[normalize-space() = "' + label + '"]]'
    elements = driver.find_elements('xpath', xpath)
    if len(elements) == 0:
        return ""
    value = elements[0].text.replace(label, "", 1)
    value = " ".join(value.split())
    return value.strip(" ;,")

# Extract the summary paragraph of the policy page
def get_summary():
    elements = driver.find_elements('xpath', '//div[@class = "policy-summary"]/p')
    if len(elements) == 0:
        return ""
    return elements[0].text.strip()

time.sleep(10)
list_links[0].click()
time.sleep(10)

print(get_field("Jurisdiction Name"))
print(get_field("Jurisdiction Type"))
print(get_field("State"))
print(get_field("Food System Category"))
print(get_summary())

# <span style="color:darkblue"> V. Loop Over All Policies </span>

<font size = "5">

Open each policy page and store the data

<font size = "3">

- Use ``` driver.get(url) ``` to open each address, wait 10 seconds (the site's crawl delay), <br>
and append a dictionary of the extracted data to a list
- If a page fails to load (for example a server timeout), we wait, try again, <br>
and record the address if it still fails
- If Chrome itself crashes, we open a new browser window and continue
- Every 25 policies the data so far is saved to ``` policy_data_backup.csv ``` in case the run is interrupted
- The full run takes about 2.5 hours. To test, set ``` num_to_scrape ``` to a small number such as 5

In [ ]:
num_to_scrape = num_links

data = []
failed_urls = []

for i in range(num_to_scrape):
    url = list_urls[i]

    loaded = False
    for attempt in range(3):
        try:
            driver.get(url)
            time.sleep(10)
            if len(driver.find_elements('xpath','//div[@class = "policy-summary"]')) > 0:
                loaded = True
                break
        except Exception as error:
            print("Problem loading", url, str(error)[:100])
            # If Chrome crashed, open a new browser window and continue
            try:
                driver.quit()
            except Exception:
                pass
            driver = webdriver.Chrome(options=options)
        time.sleep(30)

    if not loaded:
        failed_urls.append(url)
        continue

    data.append({"jurisdiction": get_field("Jurisdiction Name"),
                 "jurisdiction_type": get_field("Jurisdiction Type"),
                 "state": get_field("State"),
                 "policy_category": get_field("Food System Category"),
                 "summary": get_summary(),
                 "url": url})

    # Every 25 policies, print progress and save a backup of the data so far
    if (i + 1) % 25 == 0:
        print("Collected", i + 1, "of", num_to_scrape)
        pd.DataFrame(data).to_csv("policy_data_backup.csv", index = False, encoding = "utf-8-sig")

print("Policies that could not be loaded:", len(failed_urls))

# <span style="color:darkblue"> VI. Convert to Pandas DataFrame and Export </span>

In [ ]:
data_df = pd.DataFrame(data)

# Close the browser
driver.quit()

# Save to a CSV file
data_df.to_csv("policy_data_selenium.csv", index = False, encoding = "utf-8-sig")

# Report the observations and the collection date
print("Number of policies collected:", len(data_df))
print("Collection date:", time.strftime("%Y-%m-%d"))

# Report fields that the website leaves blank
print(data_df.eq("").sum())

data_df.head()

## Data collection notes

- Source: Healthy Food Policy Project policy database (permission obtained). The policy fields replace the product fields in the general project example.
- Restrictions: ``` robots.txt ``` allows policy pages and asks for a 10-second crawl delay, which the code follows.
- Blank values: some policies have no category, jurisdiction type, or summary on the website. These stay blank rather than being filled in.
- Output: ``` policy_data_selenium.csv ``` (the same data is also in ``` policy_data_605.csv ```).